0. 라이브러리 import

In [1]:
import os
import json
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.model_selection import train_test_split, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks

# 재현성 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

1. Google Drive에서 CSV 불러오기

In [3]:
# 구글 드라이브 마운트
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# CSV 파일 경로 지정
# 본인 구글 드라이브에 저장한 위치에 맞게 수정
csv_path = "/content/drive/MyDrive/cv_face/csv/window30sec_all.csv"

# 파일 존재 확인
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {csv_path}")

# CSV 읽기
df = pd.read_csv(csv_path)

print("CSV shape:", df.shape)
display(df.head())

print("\n라벨 분포")
print(df["label"].value_counts())

CSV shape: (6, 50)


,video_name,window_id,start_sec,end_sec,window_duration_sec,start_frame_idx,end_frame_idx,start_sample_idx,end_sample_idx,frame_count,...,yawn_count_window_max,yawn_count_window_min,no_face_duration_mean,no_face_duration_std,no_face_duration_max,no_face_duration_min,fatigue_feature_score_mean,fatigue_feature_score_std,fatigue_feature_score_max,fatigue_feature_score_min
0,WIN_20260508_14_19_02_Pro.mp4,0,0.0000,29.9719,29.972082,0,874,0,874,875,...,0.0,0.0,0.0,0.0,0.0,0.0,0.014332,0.027194,0.2159,0.0044
1,WIN_20260508_14_19_02_Pro.mp4,1,30.0062,59.9781,30.006375,875,1749,875,1749,875,...,0.0,0.0,0.0,0.0,0.0,0.0,0.046097,0.101145,0.5076,0.0030
2,WIN_20260508_14_19_02_Pro.mp4,2,60.0124,89.9843,30.006375,1750,2624,1750,2624,875,...,0.0,0.0,0.0,0.0,0.0,0.0,0.008501,0.014164,0.1376,0.0031
3,WIN_20260508_14_20_40_Pro.mp4,0,0.0000,29.9749,29.974983,0,877,0,877,878,...,0.0,0.0,0.0,0.0,0.0,0.0,0.006220,0.004006,0.0501,0.0028
4,WIN_20260508_14_20_40_Pro.mp4,1,30.0090,59.9839,30.009162,878,1755,878,1755,878,...,0.0,0.0,0.0,0.0,0.0,0.0,0.008090,0.004680,0.0557,0.0042



라벨 분포
label
drowsy    3
normal    3
Name: count, dtype: int64


2. 데이터 기본 분석

In [5]:
print("전체 컬럼 수:", len(df.columns))
print(df.columns.tolist())

# 주요 feature만 간단히 확인
important_cols = [
    "eye_closed_total_sec",
    "eye_closed_ratio",
    "eye_closed_score_mean",
    "eye_closed_duration_max",
    "jaw_open_mean",
    "jaw_open_max",
    "fatigue_feature_score_mean",
    "fatigue_feature_score_max",
    "face_detected_ratio"
]

existing_important_cols = [c for c in important_cols if c in df.columns]

summary = df.groupby("label")[existing_important_cols].mean()
display(summary)

# label별 feature 평균 차이 확인
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

diff_rows = []
for col in numeric_cols:
    drowsy_mean = df[df["label"] == "drowsy"][col].mean()
    normal_mean = df[df["label"] == "normal"][col].mean()
    diff = drowsy_mean - normal_mean
    diff_rows.append([col, drowsy_mean, normal_mean, diff, abs(diff)])

diff_df = pd.DataFrame(
    diff_rows,
    columns=["feature", "drowsy_mean", "normal_mean", "diff", "abs_diff"]
).sort_values("abs_diff", ascending=False)

display(diff_df.head(20))

전체 컬럼 수: 50
['video_name', 'window_id', 'start_sec', 'end_sec', 'window_duration_sec', 'start_frame_idx', 'end_frame_idx', 'start_sample_idx', 'end_sample_idx', 'frame_count', 'face_detected_ratio', 'eye_closed_total_sec', 'eye_closed_ratio', 'label', 'eye_blink_left_mean', 'eye_blink_left_std', 'eye_blink_left_max', 'eye_blink_left_min', 'eye_blink_right_mean', 'eye_blink_right_std', 'eye_blink_right_max', 'eye_blink_right_min', 'eye_closed_score_mean', 'eye_closed_score_std', 'eye_closed_score_max', 'eye_closed_score_min', 'eye_closed_duration_mean', 'eye_closed_duration_std', 'eye_closed_duration_max', 'eye_closed_duration_min', 'jaw_open_mean', 'jaw_open_std', 'jaw_open_max', 'jaw_open_min', 'mouth_open_duration_mean', 'mouth_open_duration_std', 'mouth_open_duration_max', 'mouth_open_duration_min', 'yawn_count_window_mean', 'yawn_count_window_std', 'yawn_count_window_max', 'yawn_count_window_min', 'no_face_duration_mean', 'no_face_duration_std', 'no_face_duration_max', 'no_face_dur

,eye_closed_total_sec,eye_closed_ratio,eye_closed_score_mean,eye_closed_duration_max,jaw_open_mean,jaw_open_max,fatigue_feature_score_mean,fatigue_feature_score_max,face_detected_ratio
label,,,,,,,,,
drowsy,3.715075,0.123841,0.320845,1.748933,0.031349,0.057813,0.022977,0.287033,1.0
normal,1.811487,0.060379,0.226333,0.273431,0.034137,0.083442,0.007744,0.054800,1.0


,feature,drowsy_mean,normal_mean,diff,abs_diff
7,end_sample_idx,1749.000000,1755.000000,-6.000000,6.000000
5,end_frame_idx,1749.000000,1755.000000,-6.000000,6.000000
6,start_sample_idx,875.000000,878.000000,-3.000000,3.000000
4,start_frame_idx,875.000000,878.000000,-3.000000,3.000000
8,frame_count,875.000000,878.000000,-3.000000,3.000000
10,eye_closed_total_sec,3.715075,1.811487,1.903588,1.903588
26,eye_closed_duration_max,1.748933,0.273431,1.475502,1.475502
25,eye_closed_duration_std,0.286398,0.025467,0.260931,0.260931
46,fatigue_feature_score_max,0.287033,0.054800,0.232233,0.232233
12,eye_blink_left_mean,0.330852,0.226912,0.103940,0.103940


3. 학습용 X, y 만들기

In [6]:
# 라벨 인코딩
# normal = 0
# drowsy = 1
label_map = {
    "normal": 0,
    "drowsy": 1
}

df["target"] = df["label"].map(label_map)

if df["target"].isna().any():
    raise ValueError("label 컬럼에 normal/drowsy 이외의 값이 있습니다.")

# 학습에서 제외할 메타데이터 컬럼
# window_id, start/end, frame index 등은 실제 졸음 특징이 아니라
# 영상 위치 정보라서 모델이 잘못 외울 수 있음
drop_cols = [
    "video_name",
    "label",
    "target",
    "window_id",
    "start_sec",
    "end_sec",
    "window_duration_sec",
    "start_frame_idx",
    "end_frame_idx",
    "start_sample_idx",
    "end_sample_idx",
    "frame_count"
]

feature_cols = [
    col for col in df.columns
    if col not in drop_cols and pd.api.types.is_numeric_dtype(df[col])
]

# NaN 처리
X_df = df[feature_cols].copy()
X_df = X_df.replace([np.inf, -np.inf], np.nan)
X_df = X_df.fillna(0)

# 값이 전부 같은 컬럼 제거
# 예: 이번 데이터에서는 face_detected_ratio, yawn_count_window 계열 등이 전부 0 또는 1이라 학습 정보가 없음
constant_cols = [
    col for col in X_df.columns
    if X_df[col].nunique() <= 1
]

X_df = X_df.drop(columns=constant_cols)

feature_cols = X_df.columns.tolist()

X = X_df.values.astype(np.float32)
y = df["target"].values.astype(np.float32)

print("사용 feature 수:", len(feature_cols))
print("제거된 상수 컬럼:", constant_cols)
print("최종 feature 목록:")
for c in feature_cols:
    print("-", c)

print("X shape:", X.shape)
print("y shape:", y.shape)

사용 feature 수: 25
제거된 상수 컬럼: ['face_detected_ratio', 'eye_closed_duration_min', 'mouth_open_duration_mean', 'mouth_open_duration_std', 'mouth_open_duration_max', 'mouth_open_duration_min', 'yawn_count_window_mean', 'yawn_count_window_std', 'yawn_count_window_max', 'yawn_count_window_min', 'no_face_duration_mean', 'no_face_duration_std', 'no_face_duration_max', 'no_face_duration_min']
최종 feature 목록:
- eye_closed_total_sec
- eye_closed_ratio
- eye_blink_left_mean
- eye_blink_left_std
- eye_blink_left_max
- eye_blink_left_min
- eye_blink_right_mean
- eye_blink_right_std
- eye_blink_right_max
- eye_blink_right_min
- eye_closed_score_mean
- eye_closed_score_std
- eye_closed_score_max
- eye_closed_score_min
- eye_closed_duration_mean
- eye_closed_duration_std
- eye_closed_duration_max
- jaw_open_mean
- jaw_open_std
- jaw_open_max
- jaw_open_min
- fatigue_feature_score_mean
- fatigue_feature_score_std
- fatigue_feature_score_max
- fatigue_feature_score_min
X shape: (6, 25)
y shape: (6,)


4. MLP 모델 함수 정의

In [7]:
def build_mlp(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(
            32,
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001)
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(
            16,
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001)
        ),
        layers.Dropout(0.2),

        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

5. 데이터가 너무 적을 때 Leave-One-Out 평가

In [8]:
# 임시 시험 데이터 기준 (데이터 충분하면 아래로 적용됨)
# 현재 CSV는 6개 샘플뿐이라 일반 train/test split의 신뢰도가 낮음.
# 그래서 샘플 1개씩 빼고 학습/테스트하는 방식으로 파이프라인 확인.

if len(df) < 30:
    print("데이터가 30개 미만이므로 Leave-One-Out 방식으로 간단 평가합니다.")

    loo = LeaveOneOut()

    y_true_list = []
    y_pred_list = []
    y_prob_list = []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = build_mlp(input_dim=X_train_scaled.shape[1])

        model.fit(
            X_train_scaled,
            y_train,
            epochs=120,
            batch_size=2,
            verbose=0
        )

        prob = model.predict(X_test_scaled, verbose=0)[0][0]
        pred = 1 if prob >= 0.5 else 0

        y_true_list.append(int(y_test[0]))
        y_pred_list.append(pred)
        y_prob_list.append(prob)

    print("Leave-One-Out Accuracy:", accuracy_score(y_true_list, y_pred_list))
    print("\nClassification Report")
    print(classification_report(
        y_true_list,
        y_pred_list,
        target_names=["normal", "drowsy"],
        zero_division=0
    ))

    print("Confusion Matrix")
    print(confusion_matrix(y_true_list, y_pred_list))

else:
    print("데이터가 충분하므로 train/test split으로 평가합니다.")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    class_weights_array = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train
    )

    class_weights = {
        int(cls): weight
        for cls, weight in zip(np.unique(y_train), class_weights_array)
    }

    model = build_mlp(input_dim=X_train_scaled.shape[1])

    es = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.2,
        epochs=300,
        batch_size=8,
        class_weight=class_weights,
        callbacks=[es],
        verbose=1
    )

    y_prob = model.predict(X_test_scaled).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report")
    print(classification_report(
        y_test,
        y_pred,
        target_names=["normal", "drowsy"],
        zero_division=0
    ))

    print("Confusion Matrix")
    print(confusion_matrix(y_test, y_pred))

    plt.figure()
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.legend()
    plt.grid(True)
    plt.title("Loss")
    plt.show()

    plt.figure()
    plt.plot(history.history["accuracy"], label="train_acc")
    plt.plot(history.history["val_accuracy"], label="val_acc")
    plt.legend()
    plt.grid(True)
    plt.title("Accuracy")
    plt.show()

데이터가 30개 미만이므로 Leave-One-Out 방식으로 간단 평가합니다.


Leave-One-Out Accuracy: 0.6666666666666666

Classification Report
              precision    recall  f1-score   support

      normal       0.67      0.67      0.67         3
      drowsy       0.67      0.67      0.67         3

    accuracy                           0.67         6
   macro avg       0.67      0.67      0.67         6
weighted avg       0.67      0.67      0.67         6

Confusion Matrix
[[2 1]
 [1 2]]


6. 최종 모델 전체 데이터로 다시 학습

In [ ]:
# 실제 저장용 모델은 전체 데이터를 사용해서 다시 학습한다.
# 단, 현재 데이터가 너무 적기 때문에 성능용이 아니라 테스트용 모델로 봐야 함.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

final_model = build_mlp(input_dim=X_scaled.shape[1])

if len(df) >= 10:
    es = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True
    )

    history = final_model.fit(
        X_scaled,
        y,
        validation_split=0.2,
        epochs=300,
        batch_size=8,
        callbacks=[es],
        verbose=1
    )
else:
    history = final_model.fit(
        X_scaled,
        y,
        epochs=150,
        batch_size=2,
        verbose=1
    )

print("최종 모델 학습 완료")

7. 모델, 스케일러, feature 목록 저장

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/cv_face/model/drowsy_mlp_model"
os.makedirs(SAVE_DIR, exist_ok=True)

model_path = os.path.join(SAVE_DIR, "drowsy_mlp.keras")
scaler_path = os.path.join(SAVE_DIR, "scaler.pkl")
feature_path = os.path.join(SAVE_DIR, "feature_columns.json")
label_path = os.path.join(SAVE_DIR, "label_map.json")

final_model.save(model_path)
joblib.dump(scaler, scaler_path)

with open(feature_path, "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

with open(label_path, "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

print("저장 완료")
print(model_path)
print(scaler_path)
print(feature_path)
print(label_path)

8. 압축 후 다운로드

In [ ]:
!zip -r drowsy_mlp_model.zip drowsy_mlp_model

files.download("drowsy_mlp_model.zip")